# Lab 06-01: Compare Vector, Keyword, and Hybrid Retrieval

**Book chapter:** Chapter 06: Enterprise Knowledge: RAG, MCP, and Integrations  
**Estimated time:** 30 minutes

## Overview

Cymbal Retail's product expert agent must answer questions phrased in everyday
language, such as boots for soggy paths, and questions that use exact product codes,
such as TRX-210. In this lab, you build three retrievers over a small catalog, measure
them on questions with known answers, and give an ADK agent the best one as a tool.

## Learning Objectives

By the end of this lab, you will be able to:

- Create embeddings with retrieval task types
- Rank documents by cosine similarity and by keyword relevance
- Combine rankings with Reciprocal Rank Fusion
- Measure retrieval quality against known answers
- Expose a retriever to an agent as a tool that cites sources

## Before You Begin

- Complete **Lab 00: Set Up the Lab Environment** in the `Appendix-C-Setup` folder

**Cost:** A few cents for embeddings and Gemini requests. No resources are created.

## How to Use This Notebook

Run the cells in order, from top to bottom, by selecting each code cell and pressing **Shift+Enter**. Read the explanation above each cell before you run it, and compare your output with the expected result that the text describes.

## Key Concepts

| Retriever | Strength | Weakness |
|---|---|---|
| **Vector** | Understands paraphrases and meaning | Can miss exact codes and names |
| **Keyword** | Matches exact terms and codes | Misses different wording |
| **Hybrid (RRF)** | Combines both strengths | One more step in the pipeline |

> **Note:** The retrievers are deliberately small so that every calculation is visible.
> Production systems use Agent Search, RAG Engine, or Vector Search, but the concepts are
> the same.

## Setup

In [ ]:
%pip install --quiet --upgrade google-adk

> **Note:** If you are using Google Colab and a message asks you to restart the
> session after the installation, click **Restart session**, and then continue
> from the next cell. You don't need to run the installation cell again.

In [ ]:
import os

# Replace with your lab project ID (see Appendix C).
PROJECT_ID = "your-project-id"  # @param {type:"string"}
# A current Gemini Flash model ID. Check the Google models page if this
# model has been retired.
MODEL = "gemini-3.8-flash"  # @param {type:"string"}

# Fall back to the environment variables set by labenv.sh (Appendix C).
if PROJECT_ID == "your-project-id":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", PROJECT_ID)
MODEL = os.environ.get("GEMINI_MODEL", MODEL)
assert PROJECT_ID != "your-project-id", "Set PROJECT_ID to your project ID."
# The lab code reads these environment variables.
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GEMINI_MODEL"] = MODEL
print("Project:", PROJECT_ID)
print("Model:  ", MODEL)

In [ ]:
import sys

if "google.colab" in sys.modules:
    # In Colab, sign in with the Google account that owns the lab project.
    from google.colab import auth
    auth.authenticate_user(project_id=PROJECT_ID)
    print("Signed in to Google Cloud from Colab.")
else:
    # In Jupyter, VS Code, or Workbench, the client libraries use Application
    # Default Credentials. If a later cell reports a credentials error, run
    # this command in a terminal and then rerun the cell:
    #   gcloud auth application-default login
    print("Using Application Default Credentials.")

## Step 1: Define the documents and the test questions

The catalog has eight short documents, some with product codes such as `TRX-210`.
`TESTS` pairs each question with the document that answers it, so you can measure
each retriever objectively.

> **Note:** `EMBED_MODEL` is the embedding model. Check the Google models page and
> update it if a newer embedding model is available.

In [ ]:
import functools
import math
import os
import re
from collections import Counter

from google import genai
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
PROJECT = os.environ["GOOGLE_CLOUD_PROJECT"]
MODEL = os.environ.get("GEMINI_MODEL", "gemini-3.8-flash")
EMBED_MODEL = "gemini-embedding-001"  # Check the models page for updates.
client = genai.Client(vertexai=True, project=PROJECT, location="global")

DOCS = {
    "TRX-200": "TRX-200 Trailhead boot: waterproof leather hiking boot "
               "with a lugged sole for muddy and wet trails.",
    "TRX-210": "TRX-210 Trailhead Wide: the TRX-200 design in a wide fit.",
    "SUM-100": "SUM-100 Summit runner: light trail running shoe with "
               "breathable mesh for dry, rocky paths.",
    "CMP-050": "CMP-050 Camp slipper: insulated slip-on for cabins and "
               "campsites, not for hiking.",
    "POL-RET": "Returns policy: unworn items can be returned within 30 "
               "days, and Gold members have 60 days.",
    "POL-DEF": "Defective items can be returned within 12 months, and "
               "Cymbal Retail pays the return shipping.",
    "CARE-01": "Care guide: reapply waterproofing spray every three months "
               "and dry boots away from heat.",
    "SIZE-01": "Sizing guide: Trailhead boots run half a size small, so "
               "choose half a size up.",
}
TESTS = [  # (question, the document that answers it)
    ("Which boots keep my feet dry on soggy, muddy paths?", "TRX-200"),
    ("Is there a TRX-210 option?", "TRX-210"),
    ("How long do Gold members have to send items back?", "POL-RET"),
    ("My sole cracked after four months. Can I get a refund?", "POL-DEF"),
    ("Should I buy a bigger size of the Trailhead?", "SIZE-01"),
]
IDS = list(DOCS)

## Step 2: Embed the documents and search by vector similarity

An **embedding** turns text into a vector whose direction reflects its meaning.
Documents are embedded once with the `RETRIEVAL_DOCUMENT` task type, and each question
with `RETRIEVAL_QUERY`. **Cosine similarity** compares the angle between two vectors,
and vector search returns the closest documents.

In [ ]:
def embed(text, task):
    """Returns the embedding of one text for a retrieval task type."""
    result = client.models.embed_content(
        model=EMBED_MODEL, contents=text,
        config=types.EmbedContentConfig(task_type=task,
                                        output_dimensionality=768))
    return result.embeddings[0].values


def cosine(a, b):
    """Cosine similarity: the angle between two vectors."""
    dot = sum(x * y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x * x for x in a)) *
                  math.sqrt(sum(y * y for y in b)))


DOC_VECTORS = {i: embed(DOCS[i], "RETRIEVAL_DOCUMENT") for i in IDS}


@functools.lru_cache
def query_vector(question):
    return tuple(embed(question, "RETRIEVAL_QUERY"))


def vector_search(question, k=3):
    q = query_vector(question)
    return sorted(IDS, key=lambda i: cosine(q, DOC_VECTORS[i]),
                  reverse=True)[:k]


print(vector_search("boots for soggy trails"))

**Expected result:** a list of three document IDs for a question about soggy trails.

## Step 3: Search by keywords, weighting rare words more heavily

Keyword search scores documents by the words they share with the question. Rare words,
such as product codes, count more than common words. This weighting is called inverse
document frequency.

In [ ]:
def words(text):
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower())


DOC_WORDS = {i: Counter(words(text)) for i, text in DOCS.items()}


def rarity(word):
    """Inverse document frequency: rare words score higher."""
    found = sum(1 for counts in DOC_WORDS.values() if word in counts)
    return math.log((len(DOCS) + 1) / (found + 0.5))


def keyword_search(question, k=3):
    asked = set(words(question))
    score = {i: sum(rarity(w) for w in asked if w in DOC_WORDS[i])
             for i in IDS}
    return sorted(IDS, key=score.get, reverse=True)[:k]


print(keyword_search("Is there a TRX-210 option?"))

**Expected result:** `TRX-210` is the first result for the product code question.

## Step 4: Combine both rankings with Reciprocal Rank Fusion (RRF)

**Reciprocal Rank Fusion (RRF)** combines rankings: each retriever gives a document
`1 / (60 + rank)` points, and the points are added. Documents that rank well in both
lists rise to the top, without needing to compare scores from different methods.

In [ ]:
def hybrid_search(question, k=3, c=60):
    scores = Counter()
    for ranking in (vector_search(question, len(IDS)),
                    keyword_search(question, len(IDS))):
        for rank, doc_id in enumerate(ranking, start=1):
            scores[doc_id] += 1 / (c + rank)
    return [doc_id for doc_id, _ in scores.most_common(k)]

## Step 5: Measure each method on the test questions

Measure how often each retriever puts the right document first, and how often it is in the top three.

In [ ]:
METHODS = {"vector": vector_search, "keyword": keyword_search,
           "hybrid": hybrid_search}
for name, search in METHODS.items():
    top1 = sum(search(q)[0] == answer for q, answer in TESTS)
    top3 = sum(answer in search(q) for q, answer in TESTS)
    print(f"{name:8} correct first result {top1}/{len(TESTS)}, "
          f"in top 3 {top3}/{len(TESTS)}")
for question, answer in TESTS:
    firsts = {name: search(question)[0] for name, search in METHODS.items()}
    print(f"\n{question}  (expected {answer})\n  {firsts}")

**What to look for:** vector search handles paraphrases, keyword search handles exact
codes, and hybrid search usually finds the right document in its top three for both.

## Step 6: Give an ADK agent the hybrid retriever as a tool

Retrieval becomes useful to an agent as a **tool**. `search_catalog` returns each passage
with its source ID so that the agent can cite it.

In [ ]:
def search_catalog(query: str) -> list[dict]:
    """Searches Cymbal Retail product and policy documents.

    Args:
        query: What to search for, in the shopper's own words.
    """
    return [{"source": d, "text": DOCS[d]} for d in hybrid_search(query)]


expert = Agent(
    name="product_expert",
    model=MODEL,
    instruction="Answer shopper questions using only search_catalog "
                "results, and cite the source IDs in square brackets. If "
                "the results do not answer the question, say so.",
    tools=[search_catalog],
)
runner = Runner(app_name="cymbal", agent=expert, auto_create_session=True,
                session_service=InMemorySessionService())
for text in ["I need boots for muddy trails, and my feet are wide.",
             "Do you sell ski jackets?"]:
    message = types.Content(role="user", parts=[types.Part(text=text)])
    for event in runner.run(user_id="shopper", session_id="s1",
                            new_message=message):
        if event.is_final_response() and event.content and \
                event.content.parts:
            print(f"\nUser: {text}\nAgent: {event.content.parts[0].text}")

**Expected result:** a recommendation of the Trailhead boots that cites `TRX-200` and `TRX-210`, and an honest answer that the catalog has no ski jackets.

## Step 7: Experiment

- Add a question of your own to `TESTS` that uses a product code and different wording
  from the catalog, and rerun Step 5.
- In Step 2, change `output_dimensionality` from 768 to 256, rerun Steps 2 to 5, and compare
  the scores. Smaller vectors are cheaper to store and search.

## Check Your Understanding

**1. Why are documents and questions embedded with different task types?**

<details><summary>Show answer</summary>

The task types optimize the vectors for their role in retrieval, which improves how well short questions match longer documents.

</details>

**2. Why does keyword search find TRX-210 so reliably?**

<details><summary>Show answer</summary>

The code is a rare word, so it receives a high weight and matches only the documents that contain it.

</details>

**3. Why does RRF combine ranks instead of scores?**

<details><summary>Show answer</summary>

Scores from different retrievers are on different scales. Ranks can be combined fairly without calibrating the scores.

</details>

## Summary

You built vector, keyword, and hybrid retrievers, measured them on test questions, and
gave an agent a retrieval tool that cites its sources. This lab created no resources, so
there is nothing to clean up.